In [1]:
import pandas as pd
import os

In [2]:
curr_dir_path = os.getcwd()
main_dir_path = os.path.abspath(os.path.join(curr_dir_path, os.pardir))
data_dir_path = os.path.join(main_dir_path, 'data')
raw_data_dir_path = os.path.join(data_dir_path, 'raw_data')
processed_data_dir_path = os.path.join(data_dir_path, 'processed_data')
cw_dir_path = os.path.join(data_dir_path, 'cw')
emissions_data_dir_path = os.path.join(raw_data_dir_path, 'emissions_data')
iea_data_dir_path = os.path.join(raw_data_dir_path, 'IEA_climate_policy_data')
wb_data_dir_path = os.path.join(raw_data_dir_path, 'wb_data')

In [3]:
wb_df = pd.read_csv(os.path.join(wb_data_dir_path, 'world_bank_indicators.csv'))
wb_df.head()

,Country,Year,GDP per capita,Trade openness (% of GDP),FDI net inflows (% of GDP),Inflation rate,Government revenue (% of GDP),Foreign debt (% of GDP),Industry size (% of GDP),Electricity consumption per capita,Population density (people per sq km),Control of corruption estimate,Energy use per capita,Fossil fuel dependency (% total energy)
0,Afghanistan,2000,174.930991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.863847,-1.271724,NaN,NaN
1,Afghanistan,2001,138.706822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.099929,NaN,NaN,NaN
2,Afghanistan,2002,178.954088,NaN,NaN,NaN,NaN,NaN,23.810127,NaN,32.776961,-1.251137,NaN,NaN
3,Afghanistan,2003,198.871116,NaN,0.022119,NaN,NaN,NaN,22.710864,NaN,34.854344,-1.344180,NaN,NaN
4,Afghanistan,2004,221.763654,NaN,-0.013397,NaN,NaN,NaN,26.226790,NaN,36.123230,-1.350647,NaN,NaN


In [4]:
columns = [
    "GDP per capita",
    "Trade openness (% of GDP)",
    "FDI net inflows (% of GDP)",
    "Inflation rate",
    "Government revenue (% of GDP)",
    "Foreign debt (% of GDP)",
    "Industry size (% of GDP)",
    "Electricity consumption per capita",
    "Population density (people per sq km)",
    "Control of corruption estimate",
    "Energy use per capita",
    "Fossil fuel dependency (% total energy)"
]

# important_columns = [
#     "Country",
#     "Year",
#     "GDP per capita",
#     "FDI net inflows (% of GDP)",
#     "Industry size (% of GDP)",
#     "Electricity consumption per capita",
#     "Population density (people per sq km)",
#     "Energy use per capita",
#     "Fossil fuel dependency (% total energy)"
# ]

important_columns = [
    "Country",
    "Year",
    "GDP per capita",
    "FDI net inflows (% of GDP)",
    "Industry size (% of GDP)",
    "Population density (people per sq km)",
]

In [5]:
wb_df_filtered = wb_df[important_columns]
wb_df_filtered

,Country,Year,GDP per capita,FDI net inflows (% of GDP),Industry size (% of GDP),Population density (people per sq km)
0,Afghanistan,2000,174.930991,NaN,NaN,30.863847
1,Afghanistan,2001,138.706822,NaN,NaN,31.099929
2,Afghanistan,2002,178.954088,NaN,23.810127,32.776961
3,Afghanistan,2003,198.871116,0.022119,22.710864,34.854344
4,Afghanistan,2004,221.763654,-0.013397,26.226790,36.123230
...,...,...,...,...,...,...
6113,Zimbabwe,2018,2271.853335,2.101721,31.037898,38.863777
6114,Zimbabwe,2019,1684.027904,0.970160,32.025947,39.476200
6115,Zimbabwe,2020,1730.413489,0.559626,32.767517,40.136714
6116,Zimbabwe,2021,1724.387731,0.871791,28.805586,40.835492


In [6]:
wb_df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6118 entries, 0 to 6117
Data columns (total 6 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Country                                6118 non-null   object 
 1   Year                                   6118 non-null   int64  
 2   GDP per capita                         5936 non-null   float64
 3   FDI net inflows (% of GDP)             5316 non-null   float64
 4   Industry size (% of GDP)               5551 non-null   float64
 5   Population density (people per sq km)  5997 non-null   float64
dtypes: float64(4), int64(1), object(1)
memory usage: 286.9+ KB


In [7]:
# Ensure the data is sorted by Country and Year
df = wb_df_filtered.sort_values(by=['Country', 'Year'])

def interpolate_and_fill(group):
    # Select only numeric columns to interpolate
    num_cols = group.select_dtypes(include='number').columns
    # Perform linear interpolation on numeric columns
    group[num_cols] = group[num_cols].interpolate(method='linear', limit_direction='both')
    # Use forward fill and backward fill directly (instead of fillna(method=...))
    group[num_cols] = group[num_cols].ffill().bfill()
    return group

# Apply the function to each country's group without including the grouping column in the operation.
df_imputed = df.groupby('Country', group_keys=False).apply(interpolate_and_fill)

# Drop rows with missing values
df_imputed = df_imputed.dropna()

# Reset the index
df_imputed = df_imputed.reset_index(drop=True)
# Sort the DataFrame by Country and Year
df_imputed = df_imputed.sort_values(by=['Country', 'Year'])

# Display the head of the imputed DataFrame
df_imputed.head()


/tmp/ipykernel_1314/3282900441.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_imputed = df.groupby('Country', group_keys=False).apply(interpolate_and_fill)


,Country,Year,GDP per capita,FDI net inflows (% of GDP),Industry size (% of GDP),Population density (people per sq km)
0,Afghanistan,2000,174.930991,0.022119,23.810127,30.863847
1,Afghanistan,2001,138.706822,0.022119,23.810127,31.099929
2,Afghanistan,2002,178.954088,0.022119,23.810127,32.776961
3,Afghanistan,2003,198.871116,0.022119,22.710864,34.854344
4,Afghanistan,2004,221.763654,-0.013397,26.226790,36.123230


In [8]:
df_imputed[df_imputed['Country'] == 'Afghanistan']

,Country,Year,GDP per capita,FDI net inflows (% of GDP),Industry size (% of GDP),Population density (people per sq km)
0,Afghanistan,2000,174.930991,0.022119,23.810127,30.863847
1,Afghanistan,2001,138.706822,0.022119,23.810127,31.099929
2,Afghanistan,2002,178.954088,0.022119,23.810127,32.776961
3,Afghanistan,2003,198.871116,0.022119,22.710864,34.854344
4,Afghanistan,2004,221.763654,-0.013397,26.226790,36.123230
5,Afghanistan,2005,254.184249,0.024181,26.812099,37.417118
6,Afghanistan,2006,274.218554,0.167907,28.210768,38.980258
7,Afghanistan,2007,376.223152,0.311634,26.882242,39.725023
8,Afghanistan,2008,381.733238,0.455360,26.915628,40.603195
9,Afghanistan,2009,452.053705,0.451889,21.897122,42.111067


In [9]:
df_imputed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 6 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Country                                5543 non-null   object 
 1   Year                                   5543 non-null   int64  
 2   GDP per capita                         5543 non-null   float64
 3   FDI net inflows (% of GDP)             5543 non-null   float64
 4   Industry size (% of GDP)               5543 non-null   float64
 5   Population density (people per sq km)  5543 non-null   float64
dtypes: float64(4), int64(1), object(1)
memory usage: 260.0+ KB


In [10]:
df_imputed.columns = df_imputed.columns.str.lower()

In [11]:
df_imputed.head()

,country,year,gdp per capita,fdi net inflows (% of gdp),industry size (% of gdp),population density (people per sq km)
0,Afghanistan,2000,174.930991,0.022119,23.810127,30.863847
1,Afghanistan,2001,138.706822,0.022119,23.810127,31.099929
2,Afghanistan,2002,178.954088,0.022119,23.810127,32.776961
3,Afghanistan,2003,198.871116,0.022119,22.710864,34.854344
4,Afghanistan,2004,221.763654,-0.013397,26.226790,36.123230


## Merge with policy data

In [12]:
policy_emissions_df = pd.read_csv(os.path.join(processed_data_dir_path, 'total_emissions_with_policies.csv'))
policy_emissions_df.head()

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,prev_year_emission,avg_emissions_prev_3_years,total_emissions
0,AFG,2000,Afghanistan,0,0.0,Low income,NaN,NaN,25.390391
1,AFG,2001,Afghanistan,0,0.0,Low income,25.390391,25.390391,23.723115
2,AFG,2002,Afghanistan,0,0.0,Low income,23.723115,24.556753,26.383509
3,AFG,2003,Afghanistan,0,0.0,Low income,26.383509,25.165672,27.071538
4,AFG,2004,Afghanistan,0,0.0,Low income,27.071538,25.726054,27.128799


In [13]:
policy_emissions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4370 entries, 0 to 4369
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   iso3                        4370 non-null   object 
 1   year                        4370 non-null   int64  
 2   country                     4370 non-null   object 
 3   policies_per_year           4370 non-null   int64  
 4   policies_last_3_years       4370 non-null   float64
 5   income_group                4370 non-null   object 
 6   prev_year_emission          4180 non-null   float64
 7   avg_emissions_prev_3_years  4180 non-null   float64
 8   total_emissions             4370 non-null   float64
dtypes: float64(4), int64(2), object(3)
memory usage: 307.4+ KB


In [14]:
df_merge = pd.merge(policy_emissions_df, df_imputed, how='inner', on=['country', 'year'])
df_merge

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,prev_year_emission,avg_emissions_prev_3_years,total_emissions,gdp per capita,fdi net inflows (% of gdp),industry size (% of gdp),population density (people per sq km)
0,AFG,2000,Afghanistan,0,0.0,Low income,NaN,NaN,25.390391,174.930991,0.022119,23.810127,30.863847
1,AFG,2001,Afghanistan,0,0.0,Low income,25.390391,25.390391,23.723115,138.706822,0.022119,23.810127,31.099929
2,AFG,2002,Afghanistan,0,0.0,Low income,23.723115,24.556753,26.383509,178.954088,0.022119,23.810127,32.776961
3,AFG,2003,Afghanistan,0,0.0,Low income,26.383509,25.165672,27.071538,198.871116,0.022119,22.710864,34.854344
4,AFG,2004,Afghanistan,0,0.0,Low income,27.071538,25.726054,27.128799,221.763654,-0.013397,26.226790,36.123230
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3629,ZWE,2018,Zimbabwe,0,0.0,Lower middle income,45.410983,46.390136,47.509033,2271.853335,2.101721,31.037898,38.863777
3630,ZWE,2019,Zimbabwe,1,0.0,Lower middle income,47.509033,46.277579,46.442562,1684.027904,0.970160,32.025947,39.476200
3631,ZWE,2020,Zimbabwe,0,1.0,Lower middle income,46.442562,46.454193,44.576343,1730.413489,0.559626,32.767517,40.136714
3632,ZWE,2021,Zimbabwe,1,1.0,Lower middle income,44.576343,46.175979,45.759664,1724.387731,0.871791,28.805586,40.835492


In [15]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3634 entries, 0 to 3633
Data columns (total 13 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   iso3                                   3634 non-null   object 
 1   year                                   3634 non-null   int64  
 2   country                                3634 non-null   object 
 3   policies_per_year                      3634 non-null   int64  
 4   policies_last_3_years                  3634 non-null   float64
 5   income_group                           3634 non-null   object 
 6   prev_year_emission                     3476 non-null   float64
 7   avg_emissions_prev_3_years             3476 non-null   float64
 8   total_emissions                        3634 non-null   float64
 9   gdp per capita                         3634 non-null   float64
 10  fdi net inflows (% of gdp)             3634 non-null   float64
 11  indu

In [16]:
# move total_emissions to the end
df_merge = df_merge[[col for col in df_merge.columns if col != 'total_emissions'] + ['total_emissions']]
df_merge.head()

,iso3,year,country,policies_per_year,policies_last_3_years,income_group,prev_year_emission,avg_emissions_prev_3_years,gdp per capita,fdi net inflows (% of gdp),industry size (% of gdp),population density (people per sq km),total_emissions
0,AFG,2000,Afghanistan,0,0.0,Low income,NaN,NaN,174.930991,0.022119,23.810127,30.863847,25.390391
1,AFG,2001,Afghanistan,0,0.0,Low income,25.390391,25.390391,138.706822,0.022119,23.810127,31.099929,23.723115
2,AFG,2002,Afghanistan,0,0.0,Low income,23.723115,24.556753,178.954088,0.022119,23.810127,32.776961,26.383509
3,AFG,2003,Afghanistan,0,0.0,Low income,26.383509,25.165672,198.871116,0.022119,22.710864,34.854344,27.071538
4,AFG,2004,Afghanistan,0,0.0,Low income,27.071538,25.726054,221.763654,-0.013397,26.226790,36.123230,27.128799


In [17]:
# Save the merged DataFrame to a CSV file
df_merge.to_csv(os.path.join(processed_data_dir_path, 'treatment_control_and_emissions_data.csv'), index=False)